# Classification ML Classique sur features TF-IDF

Ce notebook entraîne et compare 4 algorithmes ML sur une représentation **TF-IDF** des avis Yelp :
Logistic Regression, LinearSVC, Random Forest, Naive Bayes.

**Deux tâches** : Polarité (3 classes) et Score (1-5 étoiles).

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.constants import POLARITY_NAMES, SCORE_NAMES
from src.ml_utils import load_and_prepare, split_data, get_param_grids, tune_and_evaluate
from src.evaluation import plot_confusion, print_report
from src import setup_plot_style

setup_plot_style()
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Chargement et Préparation

In [ ]:
df = load_and_prepare()
data = split_data(df['text'], df['polarity'], df['stars'])
print(f"Train: {len(data['X_train'])} | Val: {len(data['X_val'])} | Test: {len(data['X_test'])}")

## 2. Vectorisation TF-IDF

Ajustement uniquement sur l'ensemble d'entraînement (pas de data leakage).

In [ ]:
vectorizer = TfidfVectorizer(max_features=10_000, min_df=5, max_df=0.7, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(data['X_train'])
X_val = vectorizer.transform(data['X_val'])
X_test = vectorizer.transform(data['X_test'])

print(f"Matrice TF-IDF : {X_train.shape}")

## 3. Classification — Score (1-5 étoiles)

In [ ]:
grids = get_param_grids()
results_sc, models_sc, preds_sc = tune_and_evaluate(
    grids, X_train, data['y_sc_train'], X_val, data['y_sc_val']
)

In [ ]:
results_sc_df = pd.DataFrame(results_sc).T
display(results_sc_df.sort_values('f1', ascending=False))

best_sc = results_sc_df['f1'].idxmax()
print(f"\nMeilleur modèle Score : {best_sc} (F1={results_sc_df.loc[best_sc, 'f1']:.4f})")

plot_confusion(data['y_sc_val'], preds_sc[best_sc], SCORE_NAMES, f'{best_sc} — Score')
print_report(data['y_sc_val'], preds_sc[best_sc], SCORE_NAMES, f'{best_sc} — Score')

## 4. Classification — Polarité (3 classes)

In [ ]:
grids_pol = get_param_grids()
results_pol, models_pol, preds_pol = tune_and_evaluate(
    grids_pol, X_train, data['y_pol_train'], X_val, data['y_pol_val']
)

In [ ]:
results_pol_df = pd.DataFrame(results_pol).T
display(results_pol_df.sort_values('f1', ascending=False))

best_pol = results_pol_df['f1'].idxmax()
print(f"\nMeilleur modèle Polarité : {best_pol} (F1={results_pol_df.loc[best_pol, 'f1']:.4f})")

plot_confusion(data['y_pol_val'], preds_pol[best_pol], POLARITY_NAMES, f'{best_pol} — Polarité')
print_report(data['y_pol_val'], preds_pol[best_pol], POLARITY_NAMES, f'{best_pol} — Polarité')

## 5. Test Final & Sauvegarde

In [ ]:
# Évaluation sur le test set
print_report(data['y_sc_test'], models_sc[best_sc].predict(X_test),
             SCORE_NAMES, f'{best_sc} — Score (Test)')
print_report(data['y_pol_test'], models_pol[best_pol].predict(X_test),
             POLARITY_NAMES, f'{best_pol} — Polarité (Test)')

# Sauvegarde
joblib.dump(models_sc[best_sc], os.path.join(MODELS_DIR, 'best_tfidf_classifier.pkl'))
joblib.dump(models_pol[best_pol], os.path.join(MODELS_DIR, 'best_tfidf_polarity.pkl'))
joblib.dump(vectorizer, os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl'))
print(f"\nModèles sauvegardés dans {MODELS_DIR}")